In [15]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [16]:
# First it was 42, but with 42, the ideal alpha was too low (0.0001). so i changed it to 0.
seed = 0
df = sns.load_dataset('tips')
df

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


In [17]:
def lasso_regression(X, y, learning_rate=0.01, n_iterations=1000, alpha=0.01):
  y = y.reshape(-1, 1)
  n = y.shape[0]
  p = X.shape[1]
  B = np.zeros(p).reshape(-1, 1)
  for i in range(n_iterations):
    gradient = 1/n * X.T @ (X@B - y) + alpha * np.sign(B)
    B -= learning_rate * gradient
  return B.reshape(-1)

def mse(y, pred):
  return np.mean((y - pred) ** 2)


In [18]:
df['sex'] = df['sex'].cat.codes
df['smoker'] = df['smoker'].cat.codes
df['time'] = df['time'].cat.codes
df = pd.get_dummies(data=df, columns=['day'])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def produce_training_data(df, features):
  X = df[features]
  y = df['tip'].to_numpy()

  X_train, X_, y_train, y_ = train_test_split(X, y, test_size=0.4, random_state=seed)
  X_test, X_validate, y_test, y_validate = train_test_split(X_, y_, test_size=0.5, random_state=seed)

  scaler = StandardScaler()
  X_train_standard = scaler.fit_transform(X_train)
  X_validate_scaled = scaler.transform(X_validate)

  assert X_train_standard.shape == X_train.shape, "Standardized X_train shape does not match X_train's shape"
  assert X_validate_scaled.shape == X_validate.shape, "Standardized X_train shape does not match X_train's shape"
  assert X_train.shape[0] == y_train.shape[0], "X train's shape does not match the y train's shape"


  X_train_standard = np.concat((np.ones(X_train_standard.shape[0]).reshape(-1, 1), X_train_standard), axis=1)
  X_validate_scaled = np.concat((np.ones(X_validate_scaled.shape[0]).reshape(-1, 1), X_validate_scaled), axis=1)

  return X_train_standard, X_validate_scaled, y_train, y_validate


In [20]:
# Feature selection with Lasso regression

features = df.columns.drop(['tip'])
X_train, X_validate, y_train, y_validate = produce_training_data(df, features)

In [21]:
def find_ideal_alpha(alphas):
  result = []
  for alpha in alphas:
    weights = lasso_regression(X_train, y_train, n_iterations=10000, alpha=alpha)
    pred = X_validate @ weights
    result.append([
      alpha,
      mse(y_validate, pred)
    ])

  result = np.array(result)
  ideal_alpha = result[result[:, 1].argmin()][0]
  return ideal_alpha

In [22]:
ideal_alpha = find_ideal_alpha([0.0001 * 10 ** p for p in range(7)])
ideal_alpha
# Lowest MSE for alpha = 0.1
# 0.01 < ideal alpha < 1

np.float64(0.1)

In [23]:
ideal_alpha = find_ideal_alpha([*[1 - 0.1 * p for p in range(10)], *[0.1 - 0.01 * p for p in range(10)]])
ideal_alpha

#RESULT: 0.1 < ideal alpha < 1

np.float64(0.29999999999999993)

In [24]:
ideal_alpha = find_ideal_alpha([ideal_alpha + 0.1 - 0.01 * p for p in range(20)])
ideal_alpha = round(ideal_alpha, 2)
ideal_alpha

#RESULT: ideal alpha = 0.31

np.float64(0.32)

In [25]:
weights = lasso_regression(X_train, y_train, n_iterations=10000, alpha=ideal_alpha)
weights = np.round(weights, 2)
weights

# NOTE: Because the solver failed to fully converge at the non-differentiable corner of the L1 penalty, some 'dead' features have weights hovering infinitesimally close to zero. 
#       I am rounding to 2 decimals to manually enforce the clear sparsity that the algorithm failed to finalize.

array([ 2.68,  0.65,  0.  , -0.  ,  0.  , -0.  , -0.  ,  0.  , -0.  ,
        0.  ])

In [26]:
weight_names = ['B' , *features]

dict(zip(weight_names, weights))

{'B': np.float64(2.68),
 'total_bill': np.float64(0.65),
 'sex': np.float64(0.0),
 'smoker': np.float64(-0.0),
 'time': np.float64(0.0),
 'size': np.float64(-0.0),
 'day_Thur': np.float64(-0.0),
 'day_Fri': np.float64(0.0),
 'day_Sat': np.float64(-0.0),
 'day_Sun': np.float64(0.0)}